In [1]:
# Integrantes:
# André Debiazzi Baldi Negrini - RM 569062
# Kaique da Silva Assis - RM 572718
# Vinicius Cristal de Oliveira - RM 572049
# Lucas Barros - RM 571528
# Renan Eskildssen - RM 571097


In [2]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # gera imagens sem precisar de interface gráfica
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [3]:
# SPRINT 4 - SERS
class Config:
    # Sistema fotovoltaico + bateria (ordem de grandeza de um kit residencial
    # com inversor híbrido Goodwe; confirme no datasheet do modelo escolhido)
    pv_kwp: float = 5.0            # potência de pico dos painéis (kWp)
    bat_kwh: float = 10.0          # capacidade útil da bateria (kWh)
    bat_pot_kw: float = 5.0        # potência máx. de carga/descarga (kW)
    eficiencia_rt: float = 0.92    # eficiência de ida e volta da bateria
    soc_min: float = 0.10          # reserva mínima (fração)

    # Tarifas (HIPÓTESES: substitua pela tarifa da sua distribuidora)
    tarifa_fora_ponta: float = 0.80   # R$/kWh
    tarifa_ponta: float = 1.60        # R$/kWh (18h-21h)
    tarifa_injecao: float = 0.35      # R$/kWh pago/creditado pela injeção
    horas_ponta: tuple = (18, 19, 20)
    horas_barata: tuple = (0, 1, 2, 3, 4, 5)
    carga_ponta_kw: float = 2.0       # estimativa de consumo médio na ponta (kW)

    # Emissões (HIPÓTESE: confira o fator médio do SIN no MCTI do ano usado)
    fator_emissao: float = 0.06       # kgCO2/kWh da rede

    seed: int = 42

In [4]:


def gerar_perfis(cfg: Config, dias: int) -> pd.DataFrame:
    """Cria séries horárias de geração FV e consumo, com variação de nuvens."""
    rng = np.random.default_rng(cfg.seed)
    idx = pd.date_range("2026-01-01", periods=dias * 24, freq="h")
    h = idx.hour.values

    # Curva solar em sino entre 6h e 18h, com fator de nuvens por dia
    sol = np.clip(np.sin(np.pi * (h - 6) / 12), 0, None)
    nuvem_dia = rng.uniform(0.35, 1.0, size=dias)           # 1 = céu limpo
    nuvem = np.repeat(nuvem_dia, 24) * rng.uniform(0.85, 1.0, size=dias * 24)
    pv = cfg.pv_kwp * 0.80 * sol * nuvem                      # 0.80 = perdas gerais

    # Consumo: base + pico de manhã + pico forte à noite
    carga = (
        0.4
        + 0.6 * np.exp(-((h - 7) ** 2) / 4)
        + 1.8 * np.exp(-((h - 19.5) ** 2) / 4)
        + rng.normal(0, 0.08, size=dias * 24)
    )
    carga = np.clip(carga, 0.2, None)

    return pd.DataFrame({"pv_kw": pv, "carga_kw": carga}, index=idx)

In [5]:
def simular(df: pd.DataFrame, cfg: Config, inteligente: bool) -> pd.DataFrame:
    eta = np.sqrt(cfg.eficiencia_rt)          # eficiência de carga e de descarga
    soc_min_kwh = cfg.soc_min * cfg.bat_kwh
    soc = soc_min_kwh
    linhas = []
    # Energia que a bateria precisa ter ao chegar na ponta
    reserva_ponta = min(cfg.bat_kwh - soc_min_kwh,
                        len(cfg.horas_ponta) * cfg.carga_ponta_kw / eta)
    antes_da_ponta = min(cfg.horas_ponta)

    sobra_hora = (df["pv_kw"] - df["carga_kw"]).clip(lower=0)
    sobra_dia = sobra_hora.resample("D").sum()

    for ts, r in df.iterrows():
        pv, carga, hora = r["pv_kw"], r["carga_kw"], ts.hour
        em_ponta = hora in cfg.horas_ponta
        tarifa = cfg.tarifa_ponta if em_ponta else cfg.tarifa_fora_ponta

        carga_bat = descarga_bat = importa = exporta = 0.0
        saldo = pv - carga

        if saldo > 0:  # sobra solar: carrega a bateria, exporta o resto
            espaco = (cfg.bat_kwh - soc) / eta
            carga_bat = min(saldo, cfg.bat_pot_kw, espaco)
            soc += carga_bat * eta
            exporta = saldo - carga_bat
        else:          # falta energia
            falta = -saldo
            # Inteligente: antes da ponta, só "segura" energia se o sol que ainda
            # vem hoje NÃO for suficiente para recompor a reserva da ponta.
            piso = soc_min_kwh
            if inteligente and hora < antes_da_ponta:
                fim_janela = ts.normalize() + pd.Timedelta(hours=antes_da_ponta - 1)
                sol_restante = sobra_hora.loc[ts + pd.Timedelta(hours=1): fim_janela].sum()
                faltaria = reserva_ponta - (soc - soc_min_kwh)
                if sol_restante * eta < faltaria:
                    piso = soc_min_kwh + reserva_ponta
            disp = max(0.0, soc - piso) * eta
            descarga_bat = min(falta, cfg.bat_pot_kw, disp)
            soc -= descarga_bat / eta
            importa = falta - descarga_bat

        if inteligente and hora in cfg.horas_barata:
            dia_seguinte = ts.normalize() + pd.Timedelta(days=1)
            sobra_amanha = sobra_dia.get(dia_seguinte, sobra_dia.iloc[-1])
            deficit = max(0.0, reserva_ponta - sobra_amanha * eta)
            alvo = soc_min_kwh + min(reserva_ponta, deficit)
            espaco = max(0.0, alvo - soc) / eta
            extra = max(0.0, min(cfg.bat_pot_kw - carga_bat, espaco))
            soc += extra * eta
            importa += extra
            carga_bat += extra

        linhas.append(
            dict(
                ts=ts, pv_kw=pv, carga_kw=carga, soc_kwh=soc,
                carga_bat_kw=carga_bat, descarga_bat_kw=descarga_bat,
                importa_kw=importa, exporta_kw=exporta,
                custo_rs=importa * tarifa - exporta * cfg.tarifa_injecao,
                co2_kg=importa * cfg.fator_emissao,
            )
        )
    return pd.DataFrame(linhas).set_index("ts")

In [6]:
def calcular_kpis(res: pd.DataFrame, nome: str) -> dict:
    consumo = res["carga_kw"].sum()
    pv = res["pv_kw"].sum()
    importa = res["importa_kw"].sum()
    exporta = res["exporta_kw"].sum()
    # Autossuficiência: quanto do consumo NÃO veio da rede
    autossuf = 1 - importa / consumo
    # Autoconsumo: quanto da geração FV foi usada localmente
    autoconsumo = 1 - exporta / pv if pv > 0 else 0
    return {
        "estrategia": nome,
        "consumo_kwh": consumo,
        "geracao_fv_kwh": pv,
        "importado_kwh": importa,
        "exportado_kwh": exporta,
        "autossuficiencia_%": 100 * autossuf,
        "autoconsumo_fv_%": 100 * autoconsumo,
        "custo_total_rs": res["custo_rs"].sum(),
        "co2_kg": res["co2_kg"].sum(),
    }

def sem_solar(df: pd.DataFrame, cfg: Config) -> dict:
    """Cenário de referência: casa sem solar nem bateria (tudo vem da rede)."""
    tarifas = np.where(df.index.hour.isin(cfg.horas_ponta),
                       cfg.tarifa_ponta, cfg.tarifa_fora_ponta)
    kwh = df["carga_kw"].values
    return {
        "estrategia": "sem_solar",
        "consumo_kwh": kwh.sum(), "geracao_fv_kwh": 0.0,
        "importado_kwh": kwh.sum(), "exportado_kwh": 0.0,
        "autossuficiencia_%": 0.0, "autoconsumo_fv_%": 0.0,
        "custo_total_rs": float((kwh * tarifas).sum()),
        "co2_kg": float(kwh.sum() * cfg.fator_emissao)
    }

In [7]:
from pathlib import Path
import matplotlib.pyplot as plt

def gerar_dashboard(pad: pd.DataFrame, intel: pd.DataFrame, kpis: pd.DataFrame,
                    cfg: Config, caminho: Path) -> None:
    fig, ax = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Gestão inteligente de energia - Solar + Bateria (Goodwe)", fontsize=14)

    # Dia de exemplo: aquele em que as duas estratégias mais se diferenciam
    dif = (intel["custo_rs"] - pad["custo_rs"]).abs().resample("D").sum()
    dia = dif.idxmax()
    d_i = intel.loc[dia: dia + pd.Timedelta(hours=23)]
    d_p = pad.loc[dia: dia + pd.Timedelta(hours=23)]
    horas = d_i.index.hour

    ax[0, 0].fill_between(horas, d_i["pv_kw"], alpha=0.4, label="Geração FV", color="orange")
    ax[0, 0].plot(horas, d_i["carga_kw"], label="Consumo", color="black")
    for h in cfg.horas_ponta:
        ax[0, 0].axvspan(h, h + 1, color="red", alpha=0.08)
    ax[0, 0].set(title=f"Geração x consumo ({dia:%d/%m}) - faixa vermelha = ponta",
                 xlabel="Hora", ylabel="kW")
    ax[0, 0].legend()

    ax[0, 1].plot(horas, d_p["soc_kwh"], label="Padrão", linestyle="--")
    ax[0, 1].plot(horas, d_i["soc_kwh"], label="Inteligente", linewidth=2)
    ax[0, 1].set(title="Energia armazenada na bateria", xlabel="Hora", ylabel="kWh")
    ax[0, 1].legend()

    k = kpis.set_index("estrategia")
    ordem = ["sem_solar", "padrao", "inteligente"]
    ax[1, 0].bar(ordem, k.loc[ordem, "custo_total_rs"],
                 color=["gray", "steelblue", "seagreen"])
    ax[1, 0].set(title="Custo de energia no período (R$)", ylabel="R$")

    ax[1, 1].bar(ordem, k.loc[ordem, "co2_kg"], color=["gray", "steelblue", "seagreen"])
    ax[1, 1].set(title="Emissões associadas à rede (kgCO2)", ylabel="kgCO2")

    plt.tight_layout()
    fig.savefig(caminho, dpi=140)
    plt.close(fig)

In [8]:
def assistente(pergunta: str, kpis: pd.DataFrame) -> str:
    """
    Versão simples e offline. Para integrar a um assistente real (Alexa,
    Google Assistant, API de um LLM), troque esta função por uma chamada ao
    serviço e envie `kpis.to_dict()` como contexto.
    """
    k = kpis.set_index("estrategia")
    p = pergunta.lower()
    if any(w in p for w in ("econom", "custo", "dinheiro")):
        eco = k.loc["sem_solar", "custo_total_rs"] - k.loc["inteligente", "custo_total_rs"]
        extra = k.loc["padrao", "custo_total_rs"] - k.loc["inteligente", "custo_total_rs"]
        return (f"No período simulado você economizou R$ {eco:.2f} em relação a não ter "
                f"solar, sendo R$ {extra:.2f} só pela estratégia inteligente.")
    if any(w in p for w in ("co2", "carbono", "emiss", "ambient")):
        red = k.loc["sem_solar", "co2_kg"] - k.loc["inteligente", "co2_kg"]
        return f"Você evitou cerca de {red:.1f} kgCO2 no período simulado."
    if any(w in p for w in ("bateria", "armazen")):
        return ("A estratégia inteligente guarda energia para o horário de ponta e "
                "carrega da rede na madrugada quando a previsão solar é fraca.")
    if "autossuf" in p or "independ" in p:
        return (f"Autossuficiência: {k.loc['inteligente', 'autossuficiencia_%']:.1f}% "
                f"(padrão: {k.loc['padrao', 'autossuficiencia_%']:.1f}%).")
    return "Posso responder sobre economia, carbono, bateria ou autossuficiência."

In [9]:
def ler_inversor_real(host: str) -> None:
    """
    Lê dados em tempo real de um inversor Goodwe na rede local, usando a
    biblioteca comunitária `goodwe` (pip install goodwe). Os nomes dos
    sensores variam por modelo: imprima `dados` e adapte ao seu equipamento.
    """
    try:
        import asyncio
        import goodwe  # type: ignore
    except ImportError:
        print("Biblioteca 'goodwe' não instalada. Rode: pip install goodwe")
        return

    async def _ler():
        inv = await goodwe.connect(host)
        dados = await inv.read_runtime_data()
        for chave, valor in dados.items():
            print(f"{chave}: {valor}")

    asyncio.run(_ler())

In [10]:
import argparse
import sys

def main() -> None:
    ap = argparse.ArgumentParser(description="Simulação Goodwe - energia inteligente")
    ap.add_argument("--dias", type=int, default=7)
    ap.add_argument("--perguntar", type=str, default=None)
    ap.add_argument("--host", type=str, default=None, help="IP do inversor Goodwe (opcional)")

    # Se estiver rodando no Jupyter/Colab, ignora os argumentos do kernel do sistema
    if "ipykernel_launcher" in sys.argv[0] or "colab_kernel_launcher" in sys.argv[0]:
        args = ap.parse_args(args=[])
    else:
        args = ap.parse_args()

    if args.host:
        ler_inversor_real(args.host)
        return

    cfg = Config()
    saida = Path("saida")
    saida.mkdir(exist_ok=True)

    df = gerar_perfis(cfg, args.dias)
    pad = simular(df, cfg, inteligente=False)
    intel = simular(df, cfg, inteligente=True)

    kpis = pd.DataFrame([
        sem_solar(df, cfg),
        calcular_kpis(pad, "padrao"),
        calcular_kpis(intel, "inteligente"),
    ]).round(2)

    pad.assign(estrategia="padrao").to_csv(saida / "resultados.csv")
    intel.assign(estrategia="inteligente").to_csv(saida / "resultados.csv", mode="a",
                                                  header=False)
    kpis.to_csv(saida / "kpis.csv", index=False)
    gerar_dashboard(pad, intel, kpis, cfg, saida / "dashboard.png")

    print(f"\n=== Resultados simulados ({args.dias} dias) ===")
    print(kpis.to_string(index=False))
    print(f"\nArquivos gerados em: {saida.resolve()}")

    pergunta = args.perguntar or "quanto economizei?"
    print(f"\nVocê: {pergunta}\nAssistente: {assistente(pergunta, kpis)}")


if __name__ == "__main__":
    main()


=== Resultados simulados (7 dias) ===
 estrategia  consumo_kwh  geracao_fv_kwh  importado_kwh  exportado_kwh  autossuficiencia_%  autoconsumo_fv_%  custo_total_rs  co2_kg
  sem_solar       127.14            0.00         127.14           0.00                0.00              0.00          133.21    7.63
     padrao       127.14          153.08          28.95          49.54               77.23             67.64            5.82    1.74
inteligente       127.14          153.08          28.95          49.54               77.23             67.64            5.82    1.74

Arquivos gerados em: /content/saida

Você: quanto economizei?
Assistente: No período simulado você economizou R$ 127.39 em relação a não ter solar, sendo R$ 0.00 só pela estratégia inteligente.
